# Task 4: Predicting Insurance Claim Amounts

## Introduction
Insurance companies need to estimate the cost of claims before setting premiums. By analyzing customer data like age, BMI, smoking habits, and family size, we can predict the medical insurance charges a customer is likely to incur.

## Problem Statement
Using the **Medical Cost Personal Dataset**, train a **Linear Regression** model to predict insurance charges. Analyze how BMI, age, and smoking status impact the charges. Evaluate using MAE and RMSE.

## Dataset
The dataset contains 1,338 records with features: age, sex, BMI, children, smoker, region, and charges.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('Libraries loaded successfully!')

## 1. Load Dataset
**Option A**: Place `insurance.csv` in the same folder and uncomment the first line.

**Option B**: Load directly from a public URL or use the synthetic dataset below.

In [ ]:
# === OPTION A: Load local CSV ===
# df = pd.read_csv('insurance.csv')

# === OPTION B: Load from URL (requires internet) ===
try:
    url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/insurance.csv'
    df = pd.read_csv(url)
    print('Dataset loaded from URL!')
except Exception:
    # === OPTION C: Synthetic dataset if URL fails ===
    print('URL failed. Using synthetic dataset...')
    np.random.seed(42)
    n = 1338
    age = np.random.randint(18, 65, n)
    sex = np.random.choice(['male', 'female'], n)
    bmi = np.random.normal(30.7, 6.1, n).clip(15.96, 53.13).round(2)
    children = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.43, 0.24, 0.18, 0.12, 0.02, 0.01])
    smoker = np.random.choice(['yes', 'no'], n, p=[0.20, 0.80])
    region = np.random.choice(['southeast', 'southwest', 'northwest', 'northeast'], n)

    charges = (
        250 * age
        + 350 * bmi
        + 400 * children
        + np.where(smoker == 'yes', 23000 + 1400 * bmi, 0)
        + np.random.normal(0, 2000, n)
    ).clip(1121).round(2)

    df = pd.DataFrame({
        'age': age, 'sex': sex, 'bmi': bmi,
        'children': children, 'smoker': smoker,
        'region': region, 'charges': charges
    })

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Dataset Understanding and Description

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
print('Smoker distribution:')
print(df['smoker'].value_counts())
print('\nAverage charges by smoker status:')
print(df.groupby('smoker')['charges'].mean().round(2))

## 3. Data Cleaning and Preparation

In [ ]:
# No missing values typically, but let's verify and encode
df_encoded = df.copy()

# Label Encoding for binary columns
le = LabelEncoder()
df_encoded['sex'] = le.fit_transform(df_encoded['sex'])         # female=0, male=1
df_encoded['smoker'] = le.fit_transform(df_encoded['smoker'])   # no=0, yes=1

# One-hot encoding for region
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

print('Encoding complete!')
print('Final columns:', df_encoded.columns.tolist())
df_encoded.head()

## 4. Exploratory Data Analysis (EDA) with Graphs

In [ ]:
# --- Insurance Charges Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribution of Insurance Charges', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Charges ($)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)

axes[1].hist(np.log(df['charges']), bins=40, color='salmon', edgecolor='black', alpha=0.8)
axes[1].set_title('Log-Transformed Charges Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Log(Charges)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)

plt.tight_layout()
plt.savefig('charges_distribution.png', dpi=150)
plt.show()

In [ ]:
# --- Smoking Status vs Charges (KEY INSIGHT) ---
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='smoker', y='charges', palette=['#2ecc71', '#e74c3c'])
plt.title('Impact of Smoking on Insurance Charges', fontsize=14, fontweight='bold')
plt.xlabel('Smoker', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.xticks([0, 1], ['Non-Smoker', 'Smoker'])
plt.tight_layout()
plt.savefig('smoking_charges.png', dpi=150)
plt.show()
print(f"Average charges — Non-smoker: ${df[df['smoker']=='no']['charges'].mean():,.0f}")
print(f"Average charges — Smoker:     ${df[df['smoker']=='yes']['charges'].mean():,.0f}")

In [ ]:
# --- Age vs Charges ---
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x='age', y='charges', hue='smoker',
    palette={'no': '#2ecc71', 'yes': '#e74c3c'}, alpha=0.7, s=60
)
plt.title('Age vs Insurance Charges (by Smoking Status)', fontsize=14, fontweight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.legend(title='Smoker')
plt.tight_layout()
plt.savefig('age_charges.png', dpi=150)
plt.show()

In [ ]:
# --- BMI vs Charges ---
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, x='bmi', y='charges', hue='smoker',
    palette={'no': '#2ecc71', 'yes': '#e74c3c'}, alpha=0.7, s=60
)
# Add BMI = 30 reference line (obesity threshold)
plt.axvline(x=30, color='navy', linestyle='--', alpha=0.7, label='BMI=30 (Obesity threshold)')
plt.title('BMI vs Insurance Charges (by Smoking Status)', fontsize=14, fontweight='bold')
plt.xlabel('BMI', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.legend(title='Smoker')
plt.tight_layout()
plt.savefig('bmi_charges.png', dpi=150)
plt.show()

In [ ]:
# --- Correlation Heatmap ---
plt.figure(figsize=(10, 7))
corr = df_encoded.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('insurance_correlation.png', dpi=150)
plt.show()

In [ ]:
# --- Average charges by region ---
region_charges = df.groupby('region')['charges'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=region_charges.index, y=region_charges.values, palette='Set2')
for p in ax.patches:
    ax.annotate(f'${p.get_height():,.0f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.title('Average Insurance Charges by Region', fontsize=14, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Average Charges ($)', fontsize=12)
plt.tight_layout()
plt.savefig('charges_region.png', dpi=150)
plt.show()

## 5. Model Training and Testing

In [ ]:
# Prepare features and target
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}')

In [ ]:
# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

print('LINEAR REGRESSION:')
print(f'  MAE  = ${lr_mae:,.2f}')
print(f'  RMSE = ${lr_rmse:,.2f}')
print(f'  R²   = {lr_r2:.4f} ({lr_r2*100:.2f}% variance explained)')

In [ ]:
# --- Random Forest Regressor (for comparison) ---
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print('RANDOM FOREST REGRESSOR:')
print(f'  MAE  = ${rf_mae:,.2f}')
print(f'  RMSE = ${rf_rmse:,.2f}')
print(f'  R²   = {rf_r2:.4f} ({rf_r2*100:.2f}% variance explained)')

## 6. Evaluation Metrics and Visualizations

In [ ]:
# Summary table
metrics_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE ($)': [round(lr_mae, 2), round(rf_mae, 2)],
    'RMSE ($)': [round(lr_rmse, 2), round(rf_rmse, 2)],
    'R² Score': [round(lr_r2, 4), round(rf_r2, 4)]
})
print('Model Performance Comparison:')
print(metrics_df.to_string(index=False))

In [ ]:
# Actual vs Predicted plots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, pred, title in zip(axes,
                             [lr_pred, rf_pred],
                             ['Linear Regression', 'Random Forest']):
    ax.scatter(y_test, pred, alpha=0.5, color='steelblue', edgecolor='white', s=40)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax.set_title(f'Actual vs Predicted — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Actual Charges ($)', fontsize=11)
    ax.set_ylabel('Predicted Charges ($)', fontsize=11)
    ax.legend()

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Residual plot for Linear Regression
residuals = y_test - lr_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(lr_pred, residuals, alpha=0.5, color='darkorange', edgecolor='white', s=40)
axes[0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0].set_title('Residuals vs Predicted (Linear Regression)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Charges ($)', fontsize=11)
axes[0].set_ylabel('Residuals ($)', fontsize=11)

axes[1].hist(residuals, bins=40, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('Residuals Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residuals ($)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].axvline(0, color='red', linestyle='--', lw=2)

plt.tight_layout()
plt.savefig('residuals.png', dpi=150)
plt.show()

In [ ]:
# Feature coefficients (Linear Regression — shows impact direction)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=True)

colors = ['#e74c3c' if c > 0 else '#3498db' for c in coef_df['Coefficient']]

plt.figure(figsize=(10, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(0, color='black', lw=1)
plt.title('Linear Regression Coefficients\n(Red=Increases Charges, Blue=Decreases Charges)',
          fontsize=13, fontweight='bold')
plt.xlabel('Coefficient Value', fontsize=11)
plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=150)
plt.show()

## 7. Conclusion and Key Insights

1. **Model Performance**:
   - **Linear Regression** explains approximately 75-80% of variance in insurance charges (R² ≈ 0.75-0.80)
   - **Random Forest** performs significantly better (R² ≈ 0.85-0.90) due to its ability to capture non-linear relationships

2. **Most Impactful Features**:
   - **Smoker status** is by far the strongest predictor — smokers pay **3-4x more** in premiums than non-smokers
   - **Age** has a strong positive effect — charges increase significantly with age
   - **BMI** positively impacts charges, especially for smokers with high BMI (the interaction is synergistic)
   - **Number of children** has a moderate positive effect

3. **Key Visualization Insights**:
   - The scatter plot of Age vs Charges shows three distinct clusters (low charges for young, medium for older non-smokers, high for smokers)
   - Southeast region tends to have slightly higher charges on average

4. **Business Recommendations**:
   - Insurance companies should weight smoking status heavily in premium calculations
   - Preventive health programs targeting smoking cessation and BMI reduction could reduce claims significantly
   - The model can be deployed to estimate premiums for new applicants in real-time